In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
import json
import subprocess
import ffmpeg

In [2]:

device = "cuda:0"
model_path = "DAMO-NLP-SG/VideoLLaMA3-7B"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    device_map={"": device},
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

conversation = [
    {"role": "system", "content": "You are a soccer coach assistant."},
    {
        "role": "user",
        "content": [
            {"type": "video", "video": {"video_path": "./annotated_vid.mp4", "fps": 1, "max_frames": 750}},
            {"type": "text", "text": "How many passes can you count?"},
        ]
    },
]

inputs = processor(
    conversation=conversation,
    add_system_prompt=True,
    add_generation_prompt=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
output_ids = model.generate(**inputs, max_new_tokens=1024)
response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
print(response)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

I can count at least five passes in the video.


In [5]:
conversation = [
    {"role": "system", "content": "You are a soccer coach assistant."},
    {
        "role": "user",
        "content": [
            {"type": "video", "video": {"video_path": "./annotated_vid2.mp4", "fps": 1, "max_frames": 750}},
            {"type": "text", "text": "How many passes are there exactly?"},
        ]
    },
]

inputs = processor(
    conversation=conversation,
    add_system_prompt=True,
    add_generation_prompt=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
output_ids = model.generate(**inputs, max_new_tokens=1024)
response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
print(response)

The video shows 10 passes.


In [6]:
# Some messages I got for video 1 (SNMOT-060), with the prompt "How many passes are there exactly?" or "How many passes can you count?"

### The commentator states there are 40 passes. -> total hallucination
### I can count at least five passes in the video. -> not much information 
### There are 10 passes. -> wrong number, the correct number is 5 or 6
### There are 14 passes. -> even wronger number

# Some messages I got for video 2 (SNMOT-061)
### The video shows 10 passes. -> the correct number is 5 or 6 again